# MicrobiomeDataSpace and MAGraph



## An overview

*   

## Setup of the environment



In [1]:
import polars as pl
import metabiome.io as mio

## Load data

In [2]:
data_dir = "/biodata/resources/day3_lab2/nar_operon_query_output"
# Load from multiple file formats
mds = mio.from_files(
    obs="{}/input/metadata".format(data_dir),           # Sample metadata
    abundance="{}/input/RPKM.json".format(data_dir),   # Gene abundance profiles
    taxonomy="{}/input/taxonomy.json".format(data_dir), # Taxonomic annotations
    functional="{}/input/FG.json".format(data_dir),    # Functional groups
    sequences="{}/input/merged.fasta".format(data_dir), # Gene sequences
    id_mapping="{}/input/id_mapping.tsv".format(data_dir) # ID cross-references
)


In [3]:
# Check data dimensions
print(f"Data shape: {mds.shape}")  # (n_samples, n_features)

Data shape: (9522, 9484)


## Profile the abundance per species per sample


In [3]:
# aggregrate by pfam_domain and taxa first
operon_abd_data = mds.groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")
operon_abd_data_diseaseMean = operon_abd_data.groupby.obs("disease_group").agg("mean")


In [6]:
operon_abd_data.var

species,pfam_domain,gc_id,msp_id,gene_category,domain,kingdom,phylum,class,order,family,genus,var
str,str,str,str,str,str,str,str,str,str,str,str,str
"""Bacteroides eggerthii""","""PF02665""","""cohort_merged__SRR8865595__k14…","""msp_046""","""core""","""Bacteria""","""Pseudomonadati""","""Bacteroidota""","""Bacteroidia""","""Bacteroidales""","""Bacteroidaceae""","""Bacteroides""","""Bacteroides eggerthii"""
"""Propionibacterium freudenreich…","""PF13247:::PF14711""","""Germany__4759__k141_114181::4:…","""msp_0616""","""core""","""Bacteria""","""Bacillati""","""Actinomycetota""","""Actinomycetes""","""Propionibacteriales""","""Propionibacteriaceae""","""Propionibacterium""","""Propionibacterium freudenreich…"
"""Actinomyces sp.""","""PF02613""","""cohort_merged__SRR2145524__k99…","""msp_271""","""core""","""Bacteria""","""Bacillati""","""Actinomycetota""","""Actinomycetes""","""Actinomycetales""","""Actinomycetaceae""","""Actinomyces""","""Actinomyces sp."""
"""Lactiplantibacillus plantarum""","""PF13247:::PF14711""","""US__11593__k141_179992::91::94…","""msp_0482""","""accessory""","""Bacteria""","""Bacillati""","""Bacillota""","""Bacilli""","""Lactobacillales""","""Lactobacillaceae""","""Lactiplantibacillus""","""Lactiplantibacillus plantarum"""
"""NA""","""PF13247""","""Israel__10088__k141_20419::2::…","""msp_1081""","""shared_accessory""","""Bacteria""","""Bacillati""","""Bacillota""","""Negativicutes""","""Veillonellales""","""Veillonellaceae""","""Veillonella""","""NA"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""Edwardsiella tarda""","""PF13247:::PF14711""","""cohort_merged__DRR171496__k141…","""msp_239""","""core""","""Bacteria""","""Pseudomonadati""","""Pseudomonadota""","""Gammaproteobacteria""","""Enterobacterales""","""Hafniaceae""","""Edwardsiella""","""Edwardsiella tarda"""
"""Veillonella infantium""","""PF02665""","""Israel__10067__k141_277285::3:…","""msp_0374""","""shared_accessory""","""Bacteria""","""Bacillati""","""Bacillota""","""Negativicutes""","""Veillonellales""","""Veillonellaceae""","""Veillonella""","""Veillonella infantium"""
"""Schaalia odontolytica""","""PF13247:::PF14711""","""Israel__10112__k141_37845::2::…","""msp_1358""","""accessory""","""Bacteria""","""Bacillati""","""Actinomycetota""","""Actinomycetes""","""Actinomycetales""","""Actinomycetaceae""","""Schaalia""","""Schaalia odontolytica"""


In [ ]:
cur_species =  "Veillonella parvula" # "Veillonella parvula", "Escherichia coli"
operon_abd_data.pl.barplot(
    feature_name=cur_species, 
    x_axis_col="disease_group",
    feature_type_col="species",
    title="Abundance of {0} by Disease Group".format(cur_species),
    color_map = {
            "Healthy": "#2166ac",
            "nonIBD": "#2166ac",
            "CD": "#b2182b",
            "UC": "#d6604d",
            "CRC": "#762a83",
            "MP": "#9970ab",
            "adenoma": "#c2a5cf",
        }
)


In [5]:
cur_species =  "Veillonella parvula" # "Veillonella parvula", "Escherichia coli"
operon_abd_data.pl.boxplot(
    feature_name=cur_species, 
    x_axis_col="disease_group",
    feature_type_col="species",
    title="Abundance of {0} by Disease Group".format(cur_species),
    color_map = {
            "Healthy": "#2166ac",
            # "nonIBD": "#2166ac",
            "CD": "#b2182b",
            "UC": "#d6604d",
            # "CRC": "#762a83",
            # "MP": "#9970ab",
            # "adenoma": "#c2a5cf",
        }
)



## Stratify by disease group

In [7]:
# operon_abd_data_CD = operon_abd_data.filter.obs(pl.col("disease_group") == "CD")
operon_abd_data_CD = mds.filter.obs(pl.col("disease_group") == "CD").groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")
operon_abd_data_UC = mds.filter.obs(pl.col("disease_group") == "UC").groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")
operon_abd_data_Healthy = mds.filter.obs(pl.col("disease_group") == "Healthy").groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")


/workspaces/course-pilot/Metabiome/src/metabiome/core/space.py:369: UserWarning:

1 filtered samples were not present in the matrix and were dropped



In [9]:
operon_abd_data_Healthy.pl.sankey(
    hierarchy_cols=["domain","phylum","class","family","genus", "species"], min_abundance=0.5
)


In [10]:
operon_abd_data_CD = mds.filter.obs(pl.col("disease_group") == "CD").groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")
operon_abd_data_CD.pl.sankey(
   hierarchy_cols=["domain","phylum","class","family","genus", "species"], min_abundance=0.5
)

In [11]:
operon_abd_data_UC = mds.filter.obs(pl.col("disease_group") == "UC").groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")
operon_abd_data_UC.pl.sankey(
   hierarchy_cols=["domain","phylum","class","family","genus", "species"], min_abundance=0.5
)

/workspaces/course-pilot/Metabiome/src/metabiome/core/space.py:369: UserWarning:

1 filtered samples were not present in the matrix and were dropped



### Task: 

